# 第二阶段：基线模型训练
训练原始 YOLOv11n，记录 mAP / FPS 作为对照组

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

model.train(
    # 核心数据与路径
    data='configs/visdrone.yaml',
    epochs=100,
    imgsz=640,  
    
    # --- 显存优化配置 ---
    batch=32,          # 8GB 显存跑 YOLO11n，16 是非常稳的，甚至可以尝试 32
    # batch=-1,        # 或者设为 -1 开启自动评估（会占用 60%-90% 显存）
    
    # --- 性能优化 ---
    workers=4,         # 建议设为 CPU 核心数的一半，Windows 系统下如果报错再改回 0
    device=0,          # 指定 5060 显卡
    
    # --- 训练稳定性 ---
    amp=True,          # 开启混合精度训练，大幅节省显存并加速
    mosaic=1.0,        # 开启 Mosaic 数据增强（对 VisDrone 这种小目标数据集非常重要）
    
    # --- 输出设置 ---
    name='baseline_v1',
    project='results',
    plots=True,
)

New https://pypi.org/project/ultralytics/8.4.46 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.10.20 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolo11n.pt, data=configs/visdrone.yaml, epochs=100, time=None, patience=100, batch=1, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=0, project=results, name=baseline_v120, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None,

train: Scanning F:\c盘空间转移\桌面\外包\yolo\data\VisDrone\labels\train.cache... 6471 images, 0 backgrounds, 0 corrupt: 100%|██████████| 6471/6471 [00:00<?, ?it/s]

train: WARNING  F:\c\\\yolo\data\VisDrone\images\train\0000137_02220_d_0000163.jpg: 1 duplicate labels removed
train: WARNING  F:\c\\\yolo\data\VisDrone\images\train\0000140_00118_d_0000002.jpg: 1 duplicate labels removed
train: WARNING  F:\c\\\yolo\data\VisDrone\images\train\9999945_00000_d_0000114.jpg: 1 duplicate labels removed
train: WARNING  F:\c\\\yolo\data\VisDrone\images\train\9999987_00000_d_0000049.jpg: 1 duplicate labels removed



val: Scanning F:\c盘空间转移\桌面\外包\yolo\data\VisDrone\labels\val.cache... 547 images, 0 backgrounds, 0 corrupt: 100%|██████████| 547/547 [00:00<?, ?it/s]


Plotting labels to results\baseline_v120\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to results\baseline_v120
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100     0.566G      1.978      4.603      1.232         86        640:   2%|▏         | 101/6471 [00:18<19:22,  5.48it/s]


KeyboardInterrupt: 

In [ ]:
# 评估基线
model = YOLO('results/baseline_v1/weights/best.pt')
metrics = model.val(data='configs/visdrone.yaml', imgsz=640)
print(f'mAP@0.5:     {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')

In [ ]:
# 可视化训练曲线
from IPython.display import Image as IPImage
IPImage('results/baseline/results.png')

In [ ]:
# 测试推理速度（FPS）
import time, cv2, numpy as np

dummy = np.zeros((640, 640, 3), dtype=np.uint8)
model = YOLO('results/baseline_v1/weights/best.pt')
# 预热
for _ in range(5):
    model(dummy, verbose=False)
# 计时
t = time.time()
N = 50
for _ in range(N):
    model(dummy, verbose=False)
fps = N / (time.time() - t)
print(f'FPS: {fps:.1f}')